# Does Iteration Alone Fix Span Edits? — Condition C2 Demo

This notebook is a **minimal-scale, runnable demo** of `method.py` from artifact `art_K8koUmGFDlLN`
("Does Iteration Alone Fix Span Edits?").

**What the experiment does.** Condition C2 uses the *same narrow, QE/oracle-span-only localization*
that conditions B and D used in the prior B/D/C1 comparison (a different artifact, `gen_art_experiment_3`),
but the repaired span is now **iteratively re-verified** by a 4-category content-invariant checker
(`named_entity`, `number_unit_date`, `negation_polarity`, `quantifier_scope`) and **re-repaired** with a
targeted, checker-detail-scoped prompt, up to `MAX_PASSES=3`, stopping as soon as a certificate is
achieved. The question this isolates: does *iteration alone* (no checker-broadened localization, unlike
condition C1) close the regression gap C1 opened, and does it improve translation quality (ΔCOMET)?

**What this notebook does differently from the full run:**
- Runs on a **tiny curated subset** (`mini_demo_data.json`, a handful of natural + injected rows per
  language pair/category) instead of the full 320-natural / 240-injected-pool sweep — so it finishes in
  minutes instead of ~70.
- `checker.py` and `llm_client.py` are copied **verbatim** into their own cells below (in the original
  artifact they are separate files imported via `sys.path`) so this notebook is self-contained.
- Real repair calls go to OpenRouter's `google/gemma-3-12b-it` (the same model the original run used) —
  you need an `OPENROUTER_API_KEY` (get one free at [openrouter.ai](https://openrouter.ai)). Cost for this
  demo's row count is a fraction of a cent (the full run cost $0.045 for 930 calls).
- COMET (`Unbabel/wmt22-cometkiwi-da`) is **not installed** here (it needs a multi-GB GPU-friendly
  download) — `try_load_comet()` already falls back to a lightweight length-ratio proxy when the `comet`
  package is unavailable, exactly as it does in the original script, so no code change was needed for that.
- The comparison against the prior run's B/D/C1 numbers is skipped gracefully: `PRIOR_RUN_PATH` points at
  a sibling artifact that isn't shipped with this demo, and the original code already handles a missing
  prior run by returning `prior_run_found: False` — again, no code change needed.

All processing logic (masking, prompts, edit-distance, the iterative repair loop, the fix/regression
scoring, the summary tables) is copied as closely as possible to the original `method.py`.


In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install everywhere)
_pip('aiohttp==3.14.3')
_pip('python-dotenv==1.2.3')
_pip('loguru==0.7.3')
_pip('psutil==7.2.2')
_pip('stanza==1.14.0')  # powers checker.py's named_entity detector (Stanza NER for ru/uk)

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'requests==2.32.4', 'tqdm==4.67.3', 'matplotlib==3.10.0')


In [ ]:
# Imports -- copied from method.py's import block, plus matplotlib for the
# visualization cell at the end. checker.py / llm_client.py are pasted as
# their own cells below instead of being imported via sys.path.insert.
from __future__ import annotations

import asyncio
import gc
import json
import random
import re
import resource
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

import psutil
from loguru import logger
import matplotlib.pyplot as plt


## Data loading

`mini_demo_data.json` is a curated subset of the same two datasets `method.py` loads from
`DATA_PATH` (`gen_art_dataset_1/data_out/full_data_out.json`): `wmt25_task3_natural` (4 rows, 2 per
language pair, each with a handful of QE-flagged spans) and `injected_error_augmentation` (8
`experimental_pool` rows — one per language pair × invariant category — plus 16
`checker_validation_heldout` rows for the checker-precision/recall step). It loads from GitHub with a
local-file fallback so this notebook works both standalone and after being published.


In [ ]:
import json, os

GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-e9bf19-isolating-what-fixes-failed-span-editing/main/round-3/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")


In [ ]:
data = load_data()
by_dataset = {ds["dataset"]: ds["examples"] for ds in data["datasets"]}
print({k: len(v) for k, v in by_dataset.items()})


## OpenRouter API key

`llm_client.py` (pasted verbatim below) reads `OPENROUTER_API_KEY` from the environment (via
`python-dotenv`, same as the original). Set it here if it isn't already in your environment — get a key
at [openrouter.ai/keys](https://openrouter.ai/keys).


In [ ]:
import os
if not os.environ.get("OPENROUTER_API_KEY"):
    try:
        from getpass import getpass
        os.environ["OPENROUTER_API_KEY"] = getpass("Enter OPENROUTER_API_KEY: ")
    except Exception:
        pass
print("OPENROUTER_API_KEY set:", bool(os.environ.get("OPENROUTER_API_KEY")))


## `checker.py` (copied verbatim)

The 4-category deterministic content-invariant checker: Stanza NER for `named_entity`, a
script-independent regex for `number_unit_date`, a per-language cue-word/negative-concord check for
`negation_polarity`, and a closed-class word list for `quantifier_scope`. It is a **localizer**, not a
correctness oracle — see its own docstring below.


In [ ]:
"""Four-category deterministic content-invariant checker for ru_RU / uk_UA.

Categories: named_entity, number_unit_date, negation_polarity, quantifier_scope.

Design (per the resource dossier, art_5ySTX4YfxqG_, Block D):
- named_entity: Stanza NER over the hypothesis text (per-language pipeline).
- number_unit_date: script-independent regex over digit sequences (+ optional
  attached unit/word token), with a boundary guard against handle-like tokens
  (e.g. @user12).
- negation_polarity: per-language cue-word list, word-boundary match, ONLY
  fires when exactly one negation marker is present in the sentence (Slavic
  negative-concord caveat documented in the dossier).
- quantifier_scope: closed-class quantifier word list per language.

The checker's role is CANDIDATE LOCALIZATION of invariant-bearing spans (it
does not itself judge cross-lingual correctness -- that judgement is
delegated to the LLM repair call in condition C1, which sees both source and
hypothesis). Precision/recall against the injected_error_augmentation
heldout fold is therefore a localization metric: does the detector's flagged
span set overlap the single known-corrupted span in that sentence. This is
an intentionally conservative precision estimate (documented in
method_out.json) because every OTHER genuine entity/number/etc. the
detector also flags in the same sentence counts against precision, even
though flagging a real invariant is exactly what a localizer should do -- the
heldout fold only labels the ONE span that was corrupted, not every
invariant-bearing span in the sentence.
"""

from __future__ import annotations

import re
from dataclasses import dataclass

CATEGORIES = (
    "named_entity",
    "number_unit_date",
    "negation_polarity",
    "quantifier_scope",
)

SUPPORTED_LANGS = ("ru_RU", "uk_UA")

# --- number/unit/date -------------------------------------------------------
# Script-independent digit-sequence regex + an optional attached unit/word
# token (letters immediately following, any script). Boundary guard: a digit
# run immediately preceded by '@' (handle-like token, e.g. @user12) is
# excluded.
_NUMBER_RE = re.compile(
    r"(?<![@\w])\d+(?:[.,]\d+)*\s?[A-Za-zА-Яа-яЁёІіЇїЄєҐґ]{0,15}\b"
)

# --- negation cue lists -------------------------------------------------
NEGATION_CUES = {
    "ru_RU": [r"не", r"нет", r"ни"],
    "uk_UA": [r"не", r"ні", r"жодн\w*"],
}

# English negation cues, used only to decide whether an ALIGNED source
# sentence plausibly carries a negation the hypothesis sentence is missing
# (see detect_negation_polarity: the injected-data negation corruption is a
# DELETION of the target-language marker, so localizing it requires noticing
# an ABSENCE relative to source, not just matching a present cue word).
_EN_NEGATION_RE = re.compile(
    r"\bnot\b|n't\b|\bnever\b|\bno\b|\bnothing\b|\bnobody\b|\bnone\b|\bneither\b|\bnor\b|\bwithout\b",
    re.IGNORECASE,
)

# --- quantifier closed-class word lists ---------------------------------
QUANTIFIER_WORDS = {
    "ru_RU": [
        "все", "всех", "всем", "всеми", "весь", "вся", "всё",
        "каждый", "каждая", "каждое", "каждые", "каждого", "каждой",
        "некоторые", "некоторых", "многие", "многих", "мало",
        "несколько", "нескольких", "любой", "любая", "любое",
        "никто", "ничто", "никакой",
    ],
    "uk_UA": [
        "всі", "весь", "вся", "все", "усі", "увесь",
        "кожен", "кожна", "кожне", "кожні", "кожного", "кожної",
        "деякі", "деяких", "багато", "багатьох", "мало",
        "декілька", "кількох", "будь-який", "будь-яка", "будь-яке",
        "ніхто", "ніщо", "жоден",
    ],
}


@dataclass
class Span:
    start: int
    end: int
    text: str
    category: str


def _regex_spans(pattern: re.Pattern, text: str, category: str) -> list[Span]:
    return [Span(m.start(), m.end(), m.group(0), category) for m in pattern.finditer(text)]


def detect_number_unit_date(hyp_text: str) -> list[Span]:
    return _regex_spans(_NUMBER_RE, hyp_text, "number_unit_date")


_SENTENCE_SPLIT_RE = re.compile(r"[^.!?\n]*[.!?\n]|[^.!?\n]+$")


def _sentence_spans(text: str) -> list[tuple[int, int]]:
    """Char (start, end) offsets for each sentence-like chunk of text (split
    on .!?/newline, terminator kept with the preceding chunk)."""
    spans = []
    for m in _SENTENCE_SPLIT_RE.finditer(text):
        if m.group(0).strip():
            spans.append((m.start(), m.end()))
    return spans or [(0, len(text))]


def detect_negation_polarity(source_text: str, hyp_text: str, lang: str) -> list[Span]:
    """Per-language cue-word match, word-boundary, applied per SENTENCE (not
    the whole multi-sentence hypothesis document): a sentence fires when it
    contains EXACTLY ONE negation marker (Slavic/Czech negative-concord
    languages allow multiple co-occurring negation markers within a sentence,
    so removing just one does not reliably flip polarity unless it is the
    only one in that sentence -- the dossier's documented caveat, scoped here
    to the sentence it actually applies to rather than the whole document).

    A second signal handles the DELETION case (the injected negation-flip
    corruption removes the target-language marker entirely, leaving nothing
    to text-match at that position): a hyp sentence with ZERO negation cues
    whose position-aligned source sentence (coarse index-proportional
    alignment -- documents are similar length, exact sentence alignment is
    not available) DOES contain an English negation cue is flagged as a
    suspected negation-deletion site (the whole hyp sentence is the span,
    since there is no token to point to)."""
    cues = NEGATION_CUES[lang]
    combined = re.compile(r"\b(?:" + "|".join(cues) + r")\b", re.IGNORECASE)
    hyp_sents = _sentence_spans(hyp_text)
    src_sents = _sentence_spans(source_text) if source_text else []
    out: list[Span] = []
    for idx, (sent_start, sent_end) in enumerate(hyp_sents):
        sentence = hyp_text[sent_start:sent_end]
        matches = list(combined.finditer(sentence))
        if len(matches) == 1:
            m = matches[0]
            out.append(Span(sent_start + m.start(), sent_start + m.end(), m.group(0), "negation_polarity"))
        elif len(matches) == 0 and src_sents:
            src_idx = min(int(idx * len(src_sents) / max(1, len(hyp_sents))), len(src_sents) - 1)
            src_st, src_en = src_sents[src_idx]
            if _EN_NEGATION_RE.search(source_text[src_st:src_en]):
                out.append(Span(sent_start, sent_end, sentence, "negation_polarity"))
    return out


def detect_quantifier_scope(hyp_text: str, lang: str) -> list[Span]:
    words = sorted(QUANTIFIER_WORDS[lang], key=len, reverse=True)
    combined = re.compile(r"\b(?:" + "|".join(re.escape(w) for w in words) + r")\b", re.IGNORECASE)
    return _regex_spans(combined, hyp_text, "quantifier_scope")


class NamedEntityDetector:
    """Wraps a per-language Stanza NER pipeline. Lazily initialized (Stanza
    pipeline construction loads model weights, so this is done once per
    language and reused for every row)."""

    def __init__(self):
        self._pipelines: dict[str, object] = {}

    def _get_pipeline(self, lang: str):
        if lang not in self._pipelines:
            import stanza

            stanza_lang = {"ru_RU": "ru", "uk_UA": "uk"}[lang]
            self._pipelines[lang] = stanza.Pipeline(
                lang=stanza_lang,
                processors="tokenize,ner",
                use_gpu=False,
                verbose=False,
                download_method=None,
            )
        return self._pipelines[lang]

    def detect(self, hyp_text: str, lang: str) -> list[Span]:
        nlp = self._get_pipeline(lang)
        doc = nlp(hyp_text)
        spans = []
        for ent in doc.ents:
            spans.append(Span(ent.start_char, ent.end_char, ent.text, "named_entity"))
        return spans


class Checker:
    """Runs the four category detectors, respecting an exclusion list of
    (lang, category) cells that failed Phase-1 validation."""

    def __init__(self, excluded_cells: set[tuple[str, str]] | None = None):
        self.excluded_cells = excluded_cells or set()
        self._ner = NamedEntityDetector()

    def detect_category(self, category: str, source_text: str, hyp_text: str, lang: str) -> list[Span]:
        if category == "named_entity":
            return self._ner.detect(hyp_text, lang)
        if category == "number_unit_date":
            return detect_number_unit_date(hyp_text)
        if category == "negation_polarity":
            return detect_negation_polarity(source_text, hyp_text, lang)
        if category == "quantifier_scope":
            return detect_quantifier_scope(hyp_text, lang)
        raise ValueError(f"Unknown category: {category}")

    def detect_all(self, source_text: str, hyp_text: str, lang: str, *, respect_exclusions: bool) -> list[Span]:
        spans: list[Span] = []
        for cat in CATEGORIES:
            if respect_exclusions and (lang, cat) in self.excluded_cells:
                continue
            spans.extend(self.detect_category(cat, source_text, hyp_text, lang))
        return spans

    def is_flagged(self, category: str, source_text: str, hyp_text: str, lang: str, start: int, end: int) -> bool:
        """True if any span from this category's detector overlaps [start, end)."""
        for s in self.detect_category(category, source_text, hyp_text, lang):
            if s.start < end and start < s.end:
                return True
        return False


## `llm_client.py` (copied verbatim)

A minimal async OpenRouter client with a live cumulative-USD cost ledger enforced against a hard cap
(`CostLedger`), used for every repair call.


In [ ]:
"""Minimal async OpenRouter client (mirrors aii-openrouter-llms' /responses
payload shape) with a live cumulative-USD cost ledger enforced against a hard
cap. Used for all condition B / D / C1 repair calls.
"""

from __future__ import annotations

import asyncio
import os
import time
from dataclasses import dataclass, field
from pathlib import Path

import aiohttp
from dotenv import load_dotenv
from loguru import logger

load_dotenv()  # notebook context: no __file__ to anchor a repo-root .env path -- OPENROUTER_API_KEY is set via the cell above instead

API_URL = "https://openrouter.ai/api/v1/responses"
MODELS_URL = "https://openrouter.ai/api/v1/models"
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")


class BudgetExceededError(RuntimeError):
    pass


@dataclass
class CostLedger:
    cap_usd: float
    spent_usd: float = 0.0
    n_calls: int = 0
    n_input_tokens: int = 0
    n_output_tokens: int = 0
    calls_log: list[dict] = field(default_factory=list)
    _lock: asyncio.Lock = field(default_factory=asyncio.Lock)

    async def add(self, cost_usd: float, input_tokens: int, output_tokens: int, meta: dict) -> None:
        async with self._lock:
            self.spent_usd += cost_usd or 0.0
            self.n_calls += 1
            self.n_input_tokens += input_tokens
            self.n_output_tokens += output_tokens
            self.calls_log.append(
                {"ts": time.time(), "cost_usd": cost_usd, "input_tokens": input_tokens,
                 "output_tokens": output_tokens, **meta}
            )
            if self.spent_usd >= self.cap_usd:
                logger.warning(f"Cost ledger at ${self.spent_usd:.4f} >= cap ${self.cap_usd:.2f}")

    def would_exceed(self, projected_extra_usd: float = 0.0) -> bool:
        return (self.spent_usd + projected_extra_usd) >= self.cap_usd

    def summary(self) -> dict:
        return {
            "spent_usd": round(self.spent_usd, 6),
            "cap_usd": self.cap_usd,
            "n_calls": self.n_calls,
            "n_input_tokens": self.n_input_tokens,
            "n_output_tokens": self.n_output_tokens,
        }


class OpenRouterClient:
    def __init__(self, ledger: CostLedger, *, max_concurrency: int = 8, timeout_s: float = 120.0):
        if not OPENROUTER_API_KEY:
            raise RuntimeError("OPENROUTER_API_KEY not set")
        self.ledger = ledger
        self.semaphore = asyncio.Semaphore(max_concurrency)
        self.timeout_s = timeout_s
        self._session: aiohttp.ClientSession | None = None
        self._pricing: dict[str, tuple[float, float]] = {}

    async def __aenter__(self):
        self._session = aiohttp.ClientSession(
            headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}", "Content-Type": "application/json"},
            timeout=aiohttp.ClientTimeout(total=self.timeout_s),
        )
        await self._load_pricing()
        return self

    async def __aexit__(self, *exc):
        if self._session:
            await self._session.close()

    async def _load_pricing(self) -> None:
        try:
            async with self._session.get(MODELS_URL) as resp:
                data = await resp.json()
            for m in data.get("data", []):
                mid = m.get("id")
                pricing = m.get("pricing", {})
                try:
                    self._pricing[mid] = (float(pricing.get("prompt", 0)), float(pricing.get("completion", 0)))
                except (TypeError, ValueError):
                    continue
        except Exception as e:
            logger.warning(f"Could not fetch OpenRouter pricing catalog: {e}")

    def _cost(self, model: str, input_tokens: int, output_tokens: int) -> float:
        price = self._pricing.get(model)
        if not price:
            return 0.0
        return input_tokens * price[0] + output_tokens * price[1]

    async def call(
        self,
        model: str,
        prompt: str,
        *,
        temperature: float = 0.0,
        max_tokens: int = 1024,
        meta: dict | None = None,
        max_retries: int = 3,
    ) -> dict:
        """Returns dict: success, text, input_tokens, output_tokens, cost_usd, error."""
        meta = meta or {}
        if self.ledger.would_exceed():
            raise BudgetExceededError(
                f"Cost ledger at ${self.ledger.spent_usd:.4f} would exceed cap ${self.ledger.cap_usd:.2f}"
            )
        payload = {
            "model": model,
            "input": prompt,
            "max_output_tokens": max_tokens,
            "temperature": temperature,
        }
        async with self.semaphore:
            last_err = None
            for attempt in range(max_retries):
                try:
                    async with self._session.post(API_URL, json=payload) as resp:
                        body = await resp.json()
                        if resp.status != 200:
                            last_err = f"HTTP {resp.status}: {str(body)[:300]}"
                            if resp.status in (429, 500, 502, 503):
                                await asyncio.sleep(2 ** attempt)
                                continue
                            break
                        text = self._extract_text(body)
                        usage = body.get("usage", {})
                        in_tok = usage.get("input_tokens", 0)
                        out_tok = usage.get("output_tokens", 0)
                        cost = self._cost(model, in_tok, out_tok)
                        await self.ledger.add(cost, in_tok, out_tok, {"model": model, **meta})
                        return {
                            "success": True, "text": text, "input_tokens": in_tok,
                            "output_tokens": out_tok, "cost_usd": cost, "error": None,
                        }
                except (aiohttp.ClientError, asyncio.TimeoutError) as e:
                    last_err = str(e)
                    await asyncio.sleep(2 ** attempt)
            logger.error(f"LLM call failed after {max_retries} attempts: {last_err}")
            return {"success": False, "text": "", "input_tokens": 0, "output_tokens": 0,
                     "cost_usd": 0.0, "error": last_err}

    @staticmethod
    def _extract_text(body: dict) -> str:
        if body.get("output_text"):
            return body["output_text"]
        for item in body.get("output", []):
            if item.get("type") == "message" and "content" in item:
                content = item["content"]
                if isinstance(content, list) and content:
                    first = content[0]
                    if isinstance(first, dict) and "text" in first:
                        return first["text"]
                elif isinstance(content, str):
                    return content
        return ""


## Config

All tunable parameters from `method.py`, set to the **smallest values that still exercise every code
path** given `mini_demo_data.json`'s size (4 natural rows, 8 injected-pool rows across the 2 language
pairs × 4 categories, 16 injected-heldout rows). `N_NATURAL_PER_PAIR` / `N_INJECTED_PER_CELL` are set to
match exactly how many rows the mini dataset has per cell, so `stratified_sample_*` (unchanged below)
selects everything rather than silently truncating. The commented-out values are the ORIGINAL full-scale
numbers from `method.py`.


In [ ]:
WORKDIR = Path(os.getcwd())
PRIOR_RUN_PATH = Path("prior_run_not_shipped_with_this_demo")  # original: gen_art_experiment_3's output dir; absent here on purpose (see intro)

SEED = 42
LANG_PAIRS = ["en-ru_RU", "en-uk_UA"]  # -> checker lang keys "ru_RU"/"uk_UA"
LANG_KEY_OF_PAIR = {"en-ru_RU": "ru_RU", "en-uk_UA": "uk_UA"}
LANG_NAME_OF_PAIR = {"en-ru_RU": "Russian", "en-uk_UA": "Ukrainian"}

N_NATURAL_PER_PAIR = 2      # original: 160
N_INJECTED_PER_CELL = 1     # original: 30

MODEL = "google/gemma-3-12b-it"
MAX_USD_BUDGET = 10.0  # hard OpenRouter-key-wide cap enforced by llm_client.CostLedger
SUB_BUDGET_USD = 2.0   # this artifact's PRE-DECLARED sub-budget (Step 4 of the plan)
COST_HALT_THRESHOLD = SUB_BUDGET_USD - 0.25  # halt before starting a new language pair near the sub-budget

MAX_CONCURRENCY = 4         # original: 8
BOOTSTRAP_ITERS = 200        # original: 1000
MAX_PASSES = 3

HYGIENE_REGEX = re.compile(
    r"__BLANK__|__[A-Z]+__|\bCorrected words\b|\bHere is the corrected\b|\bHere's the corrected\b",
    re.IGNORECASE,
)

CATEGORY_MAP_INJECTED_TO_CHECKER = {
    "named_entity_swap": "named_entity",
    "number_unit_date_alteration": "number_unit_date",
    "negation_polarity_flip": "negation_polarity",
    "quantifier_substitution": "quantifier_scope",
}

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")


## Hardware / resource limits

Copied verbatim from `method.py` (`aii-use-hardware` pattern): caps RAM and CPU time so the process
fails fast with a catchable error instead of being OOM-killed.


In [ ]:
def _container_ram_gb() -> float | None:
    for p in ["/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"]:
        try:
            v = Path(p).read_text().strip()
            if v != "max" and int(v) < 1_000_000_000_000:
                return int(v) / 1e9
        except (FileNotFoundError, ValueError):
            pass
    return None


def setup_resource_limits() -> None:
    total_ram_gb = _container_ram_gb() or psutil.virtual_memory().total / 1e9
    ram_budget_gb = min(total_ram_gb * 0.75, 20.0)
    resource.setrlimit(
        resource.RLIMIT_AS, (int(ram_budget_gb * 3 * 1024**3), int(ram_budget_gb * 3 * 1024**3))
    )
    resource.setrlimit(resource.RLIMIT_CPU, (3 * 3600, 3 * 3600))
    logger.info(f"RAM budget set to {ram_budget_gb:.1f} GB (container total {total_ram_gb:.1f} GB)")


## Phase 0 — row selection

Copied from `method.py`'s `stratified_sample_natural` / `stratified_sample_injected` /
`check_row_id_match`. `load_data()` + `index_examples()` are replaced by the `data` / `by_dataset`
already produced by the data-loading cells above (same shape, same keys) — everything downstream is
unchanged.


In [ ]:
def stratified_sample_natural(examples: list[dict], rng: random.Random) -> list[dict]:
    selected = []
    for pair in LANG_PAIRS:
        pool = [
            (i, e) for i, e in enumerate(examples)
            if e["metadata_language_pair"] == pair and e["metadata_fold"] == "experimental_pool"
        ]
        by_domain = defaultdict(list)
        for i, e in pool:
            by_domain[e["metadata_domain"]].append((i, e))
        n_domains = len(by_domain)
        per_domain = max(1, N_NATURAL_PER_PAIR // max(1, n_domains))
        picked = []
        for dom, rows in by_domain.items():
            rng.shuffle(rows)
            picked.extend(rows[:per_domain])
        rng.shuffle(picked)
        picked = picked[:N_NATURAL_PER_PAIR]
        for i, e in picked:
            row = dict(e)
            row["_row_id"] = f"wmt25_task3_natural:{i}"
            selected.append(row)
        logger.info(f"Natural subsample [{pair}]: {len(picked)} rows across {n_domains} domains")
    return selected


def stratified_sample_injected(examples: list[dict], rng: random.Random, fold: str, n_per_cell: int | None) -> list[dict]:
    selected = []
    for pair in LANG_PAIRS:
        for cat in [
            "named_entity_swap", "number_unit_date_alteration",
            "negation_polarity_flip", "quantifier_substitution",
        ]:
            cell = [
                (i, e) for i, e in enumerate(examples)
                if e["metadata_language_pair"] == pair
                and e["metadata_fold"] == fold
                and e["metadata_invariant_category"] == cat
            ]
            rng.shuffle(cell)
            if n_per_cell is not None:
                cell = cell[:n_per_cell]
            for i, e in cell:
                row = dict(e)
                row["_row_id"] = f"injected_error_augmentation:{i}"
                selected.append(row)
            logger.info(f"Injected [{fold}] subsample [{pair},{cat}]: {len(cell)} rows")
    return selected


def check_row_id_match(natural_rows: list[dict], injected_pool_rows: list[dict], injected_heldout_rows: list[dict]) -> dict:
    """Step 0 / testing_plan requirement: verify our regenerated row-ID set is
    byte-identical to B/D/C1's, using their logged selected_rows.json, rather
    than silently assuming the same seed reproduces the same draw."""
    prior_path = PRIOR_RUN_PATH / "selected_rows.json"
    result = {"prior_selected_rows_found": False, "natural_match": None, "injected_pool_match": None, "injected_heldout_match": None}
    if not prior_path.exists():
        logger.warning(f"Prior selected_rows.json not found at {prior_path} -- row-ID identity is UNVERIFIED (limitation).")
        return result
    prior = json.loads(prior_path.read_text())
    result["prior_selected_rows_found"] = True
    our_natural = sorted(r["_row_id"] for r in natural_rows)
    our_pool = sorted(r["_row_id"] for r in injected_pool_rows)
    our_held = sorted(r["_row_id"] for r in injected_heldout_rows)
    result["natural_match"] = our_natural == sorted(prior["natural_row_ids"])
    result["injected_pool_match"] = our_pool == sorted(prior["injected_pool_row_ids"])
    result["injected_heldout_match"] = our_held == sorted(prior["injected_heldout_row_ids"])
    for k in ("natural_match", "injected_pool_match", "injected_heldout_match"):
        logger.info(f"Row-ID identity check [{k}]: {result[k]}")
        if not result[k]:
            logger.warning(f"Row-ID mismatch on {k} vs prior B/D/C1 run -- comparison is population-matched, not row-matched.")
    return result


## Phase 1 — checker precision/recall validation

Copied verbatim from `method.py`. Measures the checker's localization recall/precision against the
`checker_validation_heldout` fold's known-corrupted spans; a `(lang, category)` cell that scores below
0.5 on either gets excluded from the checker's later use.


In [ ]:
def corrupted_span_offsets(row: dict) -> tuple[int, int]:
    off = row["metadata_span_offsets"]
    start = off["start"]
    end = start + len(row["metadata_corrupted_span"])
    return start, end


def validate_checker(checker: Checker, heldout_rows: list[dict]) -> dict:
    cells = defaultdict(lambda: {"tp_recall": 0, "n_rows": 0, "tp_precision": 0, "n_flags": 0})
    for row in heldout_rows:
        pair = row["metadata_language_pair"]
        lang = LANG_KEY_OF_PAIR[pair]
        cat_injected = row["metadata_invariant_category"]
        cat = CATEGORY_MAP_INJECTED_TO_CHECKER[cat_injected]
        key = (lang, cat)
        cells[key]["n_rows"] += 1
        source = row["input"]
        hyp = row["output"]
        gt_start, gt_end = corrupted_span_offsets(row)
        try:
            spans = checker.detect_category(cat, source, hyp, lang)
        except Exception as e:
            logger.error(f"Checker failed on row {row['_row_id']} cat={cat} lang={lang}: {e}")
            spans = []
        overlapped = any(s.start < gt_end and gt_start < s.end for s in spans)
        if overlapped:
            cells[key]["tp_recall"] += 1
        cells[key]["n_flags"] += len(spans)
        cells[key]["tp_precision"] += sum(1 for s in spans if s.start < gt_end and gt_start < s.end)

    results = {}
    for (lang, cat), c in cells.items():
        recall = c["tp_recall"] / c["n_rows"] if c["n_rows"] else 0.0
        precision = c["tp_precision"] / c["n_flags"] if c["n_flags"] else 0.0
        results[f"{lang}|{cat}"] = {
            "lang": lang, "category": cat, "n_rows": c["n_rows"],
            "recall": round(recall, 4), "precision": round(precision, 4),
            "n_flags_total": c["n_flags"],
        }
    return results


def excluded_cells_from_validation(validation: dict) -> set[tuple[str, str]]:
    excluded = set()
    for v in validation.values():
        if v["recall"] < 0.5 or v["precision"] < 0.5:
            excluded.add((v["lang"], v["category"]))
    return excluded


## Masking / prompts

Copied verbatim from `method.py`: the severity-skip span mask (pass 1) and the targeted-repair prompt
template used on later passes, plus the edit-distance helpers used to track convergence.


In [ ]:
def apply_severity_skip_mask(output_text: str, qe_spans: list[dict] | None) -> tuple[str, list[dict], bool]:
    if not qe_spans:
        return output_text, [], False
    has_non_minor = any(s["severity"] != "minor" for s in qe_spans)
    surviving = [s for s in qe_spans if not (has_non_minor and s["severity"] == "minor")]
    if not surviving:
        return output_text, [], False
    surviving_sorted = sorted(surviving, key=lambda s: s["start_i"])
    kept = []
    last_end = -1
    for s in surviving_sorted:
        st, en = s["start_i"], s["end_i"]
        if st < last_end or st < 0 or en > len(output_text) or en <= st:
            continue
        kept.append(s)
        last_end = en
    if not kept:
        return output_text, [], False
    masked = output_text
    for s in sorted(kept, key=lambda s: s["start_i"], reverse=True):
        masked = masked[: s["start_i"]] + "__BLANK__" + masked[s["end_i"] :]
    return masked, kept, True


MASK_PROMPT_TEMPLATE = """You are given an English source sentence and its translation into {lang_name}. Some words or phrases in the translation have been removed and replaced with the placeholder token __BLANK__ (one placeholder per removed span). Fill in each __BLANK__ with the correct {lang_name} text so the completed translation is fluent and faithful to the source sentence. Output ONLY the completed translation text with every __BLANK__ replaced by real text, and nothing else -- no explanation, no notes, no metadata, no list of corrected words.

Source (English):
{source}

Masked translation ({lang_name}):
{masked}

Completed translation ({lang_name}):"""


def build_mask_prompt(source: str, masked_text: str, lang_name: str) -> str:
    return MASK_PROMPT_TEMPLATE.format(source=source, masked=masked_text, lang_name=lang_name)


TARGETED_REPAIR_PROMPT_TEMPLATE = """You are given an English source sentence and a {lang_name} translation that you previously produced. An automated checker re-examined your translation and found it STILL has a problem in the flagged region below (category: {category}). Fix ONLY that region so it correctly corresponds to the source sentence. Do not change any other part of the translation.

Source (English):
{source}

Your previous translation ({lang_name}):
{hyp}

Still-flagged region (category={category}):
"{flagged_text}"
{flag_detail}

Output ONLY the corrected full translation text, and nothing else -- no explanation, no notes, no list of corrected words."""


def build_targeted_repair_prompt(source: str, hyp: str, category: str, flagged_text: str, flag_detail: str, lang_name: str) -> str:
    return TARGETED_REPAIR_PROMPT_TEMPLATE.format(
        source=source, hyp=hyp, category=category, flagged_text=flagged_text or "(entire sentence)",
        flag_detail=flag_detail, lang_name=lang_name,
    )


def has_leakage(text: str) -> bool:
    return bool(HYGIENE_REGEX.search(text))


def char_edit_distance(a: str, b: str) -> int:
    n, m = len(a), len(b)
    if n == 0:
        return m
    if m == 0:
        return n
    prev = list(range(m + 1))
    for i in range(1, n + 1):
        cur = [i] + [0] * m
        for j in range(1, m + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + cost)
        prev = cur
    return prev[m]


def word_edit_distance(a: str, b: str) -> int:
    aw, bw = a.split(), b.split()
    n, m = len(aw), len(bw)
    if n == 0:
        return m
    if m == 0:
        return n
    prev = list(range(m + 1))
    for i in range(1, n + 1):
        cur = [i] + [0] * m
        for j in range(1, m + 1):
            cost = 0 if aw[i - 1] == bw[j - 1] else 1
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + cost)
        prev = cur
    return prev[m]


def edit_volume(original: str, edited: str) -> float:
    n = max(1, len(original.split()))
    return word_edit_distance(original, edited) / n


## Oracle span, invariant multiset, fix/regression scoring

Copied verbatim. `oracle_qe_span` gives injected rows the corrupted-span's character offsets as the
localization target (identical to conditions B/D). `negation_polarity_flip` corruptions are DELETIONS
(no contiguous substring survives to mask), inherited as a known limitation exactly as B/D documented
it. `invariant_multiset` / `compute_fix_and_regression` are the SAME ground-truth definitions the prior
run used for `fixed` / `true_regression`, reused here (not reinvented) as the certificate C2's iteration
loop verifies against.


In [ ]:
def oracle_qe_span(row: dict) -> list[dict]:
    start, end = corrupted_span_offsets(row)
    return [{"start_i": start, "end_i": end, "severity": "major"}]


def is_negation_deletion_row(row: dict) -> bool:
    return row["metadata_invariant_category"] == "negation_polarity_flip" and row["metadata_corrupted_span"] == ""


def invariant_multiset(source_text: str, text: str, lang: str, checker: Checker) -> Counter:
    spans = checker.detect_all(source_text, text, lang, respect_exclusions=True)
    return Counter((s.category, s.text.strip().lower()) for s in spans)


def compute_fix_and_regression(row: dict, repaired_output: str, lang: str, checker: Checker) -> dict:
    source = row["input"]
    corrupted_span = row["metadata_corrupted_span"]
    clean_target = row["metadata_clean_target_text"]
    fixed = corrupted_span not in repaired_output if corrupted_span else None
    gold_ms = invariant_multiset(source, clean_target, lang, checker)
    repaired_ms = invariant_multiset(source, repaired_output, lang, checker)
    lost = sum((gold_ms - repaired_ms).values())
    corrupted_ms = invariant_multiset(source, row["output"], lang, checker)
    lost_baseline = sum((gold_ms - corrupted_ms).values())
    return {
        "fixed": fixed,
        "invariants_lost_vs_gold": int(lost),
        "invariants_lost_vs_gold_baseline": int(lost_baseline),
        "true_regression": lost > lost_baseline,
    }


def splice(original_text: str, start: int, end: int, replacement: str) -> str:
    return original_text[:start] + replacement + original_text[end:]


## Step 3 — C2 iterative repair loop

The core of the experiment, copied verbatim from `method.py`. `run_c2_injected_row` / `run_c2_natural_row`
run the mask-then-iterate loop per row (mask on pass 1, targeted checker-detail-scoped prompt on later
passes, stopping the moment a certificate is achieved or `MAX_PASSES` is hit); `process_*_row` wrap them
with error handling; `run_pipeline` drives the whole sweep per language pair with the pre-declared
sub-budget hard stop.


In [ ]:
async def run_c2_injected_row(row: dict, client: OpenRouterClient, checker: Checker,
                                excluded_cells: set[tuple[str, str]], source: str, lang: str,
                                lang_name: str, row_id: str) -> dict:
    cat_injected = row["metadata_invariant_category"]
    cat_checker = CATEGORY_MAP_INJECTED_TO_CHECKER[cat_injected]
    corrupted_hyp = row["output"]

    if is_negation_deletion_row(row):
        return {
            "c2_attempted": False, "skip_reason": "negation_deletion_zero_width_span",
            "output": corrupted_hyp, "n_passes": 0, "n_llm_calls": 0, "pass_log": [],
            "certificate_achieved": None, "fixed": None, "true_regression": None,
        }

    start, end = corrupted_span_offsets(row)
    checker_covers = (lang, cat_checker) not in excluded_cells
    current_span_text = corrupted_hyp[start:end]
    current_full = corrupted_hyp
    n_llm_calls = 0
    pass_log: list[dict] = []
    certificate_achieved = False

    for pass_num in range(1, MAX_PASSES + 1):
        if pass_num == 1:
            masked = splice(current_full, start, end, "__BLANK__")
            prompt = build_mask_prompt(source, masked, lang_name)
        else:
            flag_detail = f"The checker's {cat_checker} detector still flags this region after your previous fix."
            prompt = build_targeted_repair_prompt(source, current_full, cat_checker, current_span_text, flag_detail, lang_name)
        result = await client.call(MODEL, prompt, temperature=0.0, max_tokens=1536,
                                     meta={"row_id": row_id, "condition": "C2", "pass": pass_num})
        n_llm_calls += 1
        repaired_full = result["text"].strip() if result["success"] and result["text"].strip() else current_full

        verdict = compute_fix_and_regression(row, repaired_full, lang, checker)
        ev = char_edit_distance(current_full, repaired_full)
        # Best-effort tracking of the edited span's own text for the trend
        # signal (Step 5's char_length_trend): the span position can drift as
        # the sentence is rewritten, so track the whole-sentence length
        # delta from the pre-repair corrupted sentence as the stable proxy.
        span_char_len = len(repaired_full) - len(corrupted_hyp) + (end - start)
        passed = bool(verdict["fixed"]) and not verdict["true_regression"]
        pass_log.append({
            "pass_num": pass_num, "checker_passed": passed, "edit_distance_chars": ev,
            "span_char_length_proxy": span_char_len, "fixed": verdict["fixed"],
            "true_regression": verdict["true_regression"], "error": result["error"],
            "input_tokens": result["input_tokens"], "output_tokens": result["output_tokens"],
        })
        current_full = repaired_full
        current_span_text = repaired_full[start:end] if end <= len(repaired_full) else repaired_full
        if passed:
            certificate_achieved = True
            break
        if not checker_covers:
            break  # uncheckable_category_single_pass_only

    final = compute_fix_and_regression(row, current_full, lang, checker)
    return {
        "c2_attempted": True, "skip_reason": None, "output": current_full,
        "n_passes": len(pass_log), "n_llm_calls": n_llm_calls, "pass_log": pass_log,
        "certificate_achieved": certificate_achieved,
        "checker_covered_category": checker_covers,
        "fixed": final["fixed"], "true_regression": final["true_regression"],
        "leakage": has_leakage(current_full),
    }


async def run_c2_natural_row(row: dict, client: OpenRouterClient, checker: Checker,
                               excluded_cells: set[tuple[str, str]], source: str, lang: str,
                               lang_name: str, row_id: str) -> dict:
    hyp = row["output"]
    qe_spans = row.get("metadata_qe_flagged_spans")
    masked_text, surviving, did_mask = apply_severity_skip_mask(hyp, qe_spans)
    if not did_mask:
        return {
            "output": hyp, "n_passes": 0, "n_llm_calls": 0, "pass_log": [],
            "certificate_achieved": None, "no_op": True, "leakage": False,
        }
    span_bounds = (min(s["start_i"] for s in surviving), max(s["end_i"] for s in surviving))
    current_full = hyp
    n_llm_calls = 0
    pass_log: list[dict] = []
    certificate_achieved = False
    checker_covers_any = any((lang, cat) not in excluded_cells for cat in CATEGORIES)

    for pass_num in range(1, MAX_PASSES + 1):
        if pass_num == 1:
            prompt = build_mask_prompt(source, masked_text, lang_name)
        else:
            flagged_now = checker.detect_all(source, current_full, lang, respect_exclusions=True)
            overlapping = [f for f in flagged_now if f.start < span_bounds[1] and span_bounds[0] < f.end]
            if not overlapping:
                break
            f0 = overlapping[0]
            flag_detail = f"{len(overlapping)} checker flag(s) remain in/near the edited region."
            prompt = build_targeted_repair_prompt(source, current_full, f0.category, f0.text, flag_detail, lang_name)
        result = await client.call(MODEL, prompt, temperature=0.0, max_tokens=1536,
                                     meta={"row_id": row_id, "condition": "C2", "pass": pass_num})
        n_llm_calls += 1
        repaired_full = result["text"].strip() if result["success"] and result["text"].strip() else current_full
        ev = char_edit_distance(current_full, repaired_full)
        flagged_after = checker.detect_all(source, repaired_full, lang, respect_exclusions=True)
        overlapping_after = [f for f in flagged_after if f.start < span_bounds[1] and span_bounds[0] < f.end]
        passed = len(overlapping_after) == 0
        pass_log.append({
            "pass_num": pass_num, "checker_passed": passed, "edit_distance_chars": ev,
            "span_char_length_proxy": len(repaired_full) - len(hyp) + (span_bounds[1] - span_bounds[0]),
            "n_flags_remaining": len(overlapping_after), "error": result["error"],
            "input_tokens": result["input_tokens"], "output_tokens": result["output_tokens"],
        })
        current_full = repaired_full
        if passed:
            certificate_achieved = True
            break
        if not checker_covers_any:
            break

    return {
        "output": current_full, "n_passes": len(pass_log), "n_llm_calls": n_llm_calls,
        "pass_log": pass_log, "certificate_achieved": certificate_achieved, "no_op": False,
        "leakage": has_leakage(current_full), "edit_volume": round(edit_volume(hyp, current_full), 4),
    }


async def process_injected_row(row: dict, client: OpenRouterClient, checker: Checker, excluded_cells: set) -> dict:
    pair = row["metadata_language_pair"]
    lang = LANG_KEY_OF_PAIR[pair]
    lang_name = LANG_NAME_OF_PAIR[pair]
    source = row["input"]
    row_id = row["_row_id"]
    out = {"row_id": row_id, "language_pair": pair, "category": row["metadata_invariant_category"]}
    try:
        c2 = await run_c2_injected_row(row, client, checker, excluded_cells, source, lang, lang_name, row_id)
        out["C2"] = c2
    except BudgetExceededError:
        raise
    except Exception as e:
        logger.error(f"C2 failed on injected row {row_id}: {e}")
        out["C2"] = {"error": str(e), "c2_attempted": False, "skip_reason": "exception"}
    return out


async def process_natural_row(row: dict, client: OpenRouterClient, checker: Checker, excluded_cells: set) -> dict:
    pair = row["metadata_language_pair"]
    lang = LANG_KEY_OF_PAIR[pair]
    lang_name = LANG_NAME_OF_PAIR[pair]
    source = row["input"]
    hyp = row["output"]
    row_id = row["_row_id"]
    out = {"row_id": row_id, "language_pair": pair, "domain": row["metadata_domain"], "original": hyp}
    try:
        c2 = await run_c2_natural_row(row, client, checker, excluded_cells, source, lang, lang_name, row_id)
        out["C2"] = c2
    except BudgetExceededError:
        raise
    except Exception as e:
        logger.error(f"C2 failed on natural row {row_id}: {e}")
        out["C2"] = {"error": str(e)}
    return out


async def run_pipeline(natural_rows: list[dict], injected_rows: list[dict], checker: Checker,
                         excluded_cells: set, ledger: CostLedger) -> tuple[list[dict], list[dict]]:
    """Step 4: HARD STOP the sweep if cumulative spend hits SUB_BUDGET_USD,
    processed per language pair so a halt still leaves complete coverage for
    at least one pair (mirrors gen_art_experiment_3's fallback_plan)."""
    natural_results, injected_results = [], []
    async with OpenRouterClient(ledger, max_concurrency=MAX_CONCURRENCY) as client:
        for pair in LANG_PAIRS:
            if ledger.spent_usd >= COST_HALT_THRESHOLD:
                logger.warning(f"Halting before language pair {pair}: cost ledger ${ledger.spent_usd:.4f} >= sub-budget threshold ${COST_HALT_THRESHOLD:.2f}")
                break
            nat_pair = [r for r in natural_rows if r["metadata_language_pair"] == pair]
            inj_pair = [r for r in injected_rows if r["metadata_language_pair"] == pair]
            logger.info(f"Processing {pair}: {len(nat_pair)} natural, {len(inj_pair)} injected rows")

            nat_batches = await asyncio.gather(
                *[process_natural_row(r, client, checker, excluded_cells) for r in nat_pair], return_exceptions=True
            )
            for r in nat_batches:
                if isinstance(r, BudgetExceededError):
                    logger.warning("Budget exceeded during natural-row processing; stopping.")
                    break
                if isinstance(r, Exception):
                    logger.error(f"Unhandled error in natural row: {r}")
                    continue
                natural_results.append(r)

            inj_batches = await asyncio.gather(
                *[process_injected_row(r, client, checker, excluded_cells) for r in inj_pair], return_exceptions=True
            )
            for r in inj_batches:
                if isinstance(r, BudgetExceededError):
                    logger.warning("Budget exceeded during injected-row processing; stopping.")
                    break
                if isinstance(r, Exception):
                    logger.error(f"Unhandled error in injected row: {r}")
                    continue
                injected_results.append(r)

            logger.info(f"Cost ledger after {pair}: {ledger.summary()}")
            if ledger.spent_usd >= SUB_BUDGET_USD:
                logger.warning(f"Sub-budget ${SUB_BUDGET_USD:.2f} reached; stopping sweep after {pair}.")
                break
    return natural_results, injected_results


## Step 4 — pre-sweep cost estimation

Copied verbatim. Probes a subset of rows at pass-1-only cost to project a worst-case for the full
sweep before committing to it, against the pre-declared `SUB_BUDGET_USD`.


In [ ]:
async def estimate_cost(natural_rows: list[dict], injected_rows: list[dict], checker: Checker,
                          excluded_cells: set) -> dict:
    subset_nat = natural_rows[:10]
    subset_inj = injected_rows[:10]
    probe_ledger = CostLedger(cap_usd=1.0)  # small local cap just for the probe
    t0 = time.time()
    async with OpenRouterClient(probe_ledger, max_concurrency=MAX_CONCURRENCY) as client:
        nat_out = await asyncio.gather(
            *[process_natural_row(r, client, checker, excluded_cells) for r in subset_nat], return_exceptions=True
        )
        inj_out = await asyncio.gather(
            *[process_injected_row(r, client, checker, excluded_cells) for r in subset_inj], return_exceptions=True
        )
    elapsed = time.time() - t0
    n_probe_rows = len(subset_nat) + len(subset_inj)
    cost_per_row_pass1 = probe_ledger.spent_usd / max(1, n_probe_rows)
    # worst case: every eligible row iterates MAX_PASSES times at this per-call rate
    n_eligible_full = len(natural_rows) + sum(1 for r in injected_rows if not is_negation_deletion_row(r))
    projected_worst_case = cost_per_row_pass1 * MAX_PASSES * n_eligible_full
    projected_typical = cost_per_row_pass1 * 1.5 * n_eligible_full  # most rows pass in <=2 passes empirically at low leakage rates
    result = {
        "n_probe_rows": n_probe_rows, "probe_cost_usd": round(probe_ledger.spent_usd, 6),
        "probe_elapsed_s": round(elapsed, 1), "cost_per_row_pass1_usd": round(cost_per_row_pass1, 6),
        "n_eligible_rows_full_sweep": n_eligible_full,
        "projected_worst_case_usd": round(projected_worst_case, 4),
        "projected_typical_usd": round(projected_typical, 4),
        "sub_budget_usd": SUB_BUDGET_USD, "decision": None, "max_passes_used": MAX_PASSES,
        "probe_natural_examples": [r for r in nat_out if not isinstance(r, Exception)][:3],
        "probe_injected_examples": [r for r in inj_out if not isinstance(r, Exception)][:3],
    }
    if projected_worst_case > SUB_BUDGET_USD:
        result["decision"] = (
            f"Projected worst-case ${projected_worst_case:.4f} exceeds sub-budget ${SUB_BUDGET_USD:.2f}; "
            "would reduce MAX_PASSES or row count per fallback_plan (NOT triggered this run -- see actual figures)."
        )
    else:
        result["decision"] = (
            f"Projected worst-case ${projected_worst_case:.4f} is within sub-budget ${SUB_BUDGET_USD:.2f}; "
            "proceeding with the full sweep at MAX_PASSES=3, all eligible rows, no reduction needed."
        )
    logger.info(f"Cost estimate: {json.dumps({k: v for k, v in result.items() if not k.startswith('probe_') or 'examples' not in k}, indent=2, default=str)}")
    return result


## Phase 5 — COMET scoring (with graceful proxy fallback)

Copied verbatim, including the fallback: `try_load_comet()` already catches an import/download failure
and falls back to `proxy_qe_score` (a lightweight length-ratio proxy), which is exactly what happens in
this notebook since the `comet`/`unbabel-comet` package is not installed here (its checkpoint download
is multi-GB and needs a GPU-friendly environment) — no code change was needed to make that path work.
`bootstrap_ci` (also copied verbatim) computes the 95% CI on ΔCOMET.


In [ ]:
def try_load_comet():
    try:
        import comet

        logger.info("Downloading/loading Unbabel/wmt22-cometkiwi-da checkpoint...")
        model_path = comet.download_model("Unbabel/wmt22-cometkiwi-da")
        model = comet.load_from_checkpoint(model_path)
        logger.info("COMET checkpoint loaded successfully.")
        return model
    except Exception as e:
        logger.warning(f"COMET (Unbabel/wmt22-cometkiwi-da) unavailable: {e}. Falling back to a non-gated proxy.")
        return None


def comet_score_batch(model, sources: list[str], hyps: list[str]) -> list[float]:
    import torch

    gpus = 1 if torch.cuda.is_available() else 0
    data = [{"src": s, "mt": h} for s, h in zip(sources, hyps)]
    out = model.predict(data, batch_size=16, gpus=gpus, progress_bar=False)
    return list(out.scores)


def proxy_qe_score(source: str, hyp: str) -> float:
    ratio = len(hyp.strip()) / max(1, len(source.strip()))
    return max(0.0, 1.0 - abs(ratio - 1.0))


def bootstrap_ci(values: list[float], iters: int, rng: random.Random) -> tuple[float, float, float]:
    if not values:
        return (0.0, 0.0, 0.0)
    n = len(values)
    mean = sum(values) / n
    boots = []
    for _ in range(iters):
        sample = [values[rng.randrange(n)] for _ in range(n)]
        boots.append(sum(sample) / n)
    boots.sort()
    lo = boots[int(0.025 * iters)]
    hi = boots[min(iters - 1, int(0.975 * iters))]
    return (mean, lo, hi)


## Step 5/6 — summarize + B/D/C1-vs-C2 comparison table

Copied verbatim. `summarize_c2` builds the pooled and per-category/per-language breakdowns (never
pooled-only) plus the descriptive CEGIS-narrowing signal; `build_comparison_table` loads the prior
run's `method_out.json` for a side-by-side B/D/C1-vs-C2 table — already handles a missing prior run
(`prior_run_found: False`), which is exactly what happens here since that sibling artifact isn't
shipped with this demo.


In [ ]:
NEGATION_LABEL = "N/A (structural non-attempt)"


def summarize_c2(natural_results: list[dict], injected_results: list[dict], natural_comet: dict,
                   rng: random.Random) -> dict:
    per_pair = defaultdict(dict)
    for pair in LANG_PAIRS:
        deltas = natural_comet.get(pair, [])
        mean, lo, hi = bootstrap_ci(deltas, BOOTSTRAP_ITERS, rng)
        evs = [r["C2"]["edit_volume"] for r in natural_results
               if r["language_pair"] == pair and "edit_volume" in r.get("C2", {})]
        calls = [r["C2"].get("n_llm_calls", 0) for r in natural_results if r["language_pair"] == pair and "C2" in r]
        passes = [r["C2"].get("n_passes", 0) for r in natural_results if r["language_pair"] == pair and "C2" in r]
        per_pair[pair] = {
            "delta_comet_mean": mean, "delta_comet_ci95": [lo, hi], "n_scored": len(deltas),
            "edit_volume_mean": round(sum(evs) / len(evs), 4) if evs else None,
            "n_llm_calls_mean": round(sum(calls) / len(calls), 4) if calls else None,
            "n_passes_mean": round(sum(passes) / len(passes), 4) if passes else None,
        }
    all_deltas = [v for pair in LANG_PAIRS for v in natural_comet.get(pair, [])]
    mean, lo, hi = bootstrap_ci(all_deltas, BOOTSTRAP_ITERS, rng)
    all_calls = [r["C2"].get("n_llm_calls", 0) for r in natural_results if "C2" in r]
    all_passes = [r["C2"].get("n_passes", 0) for r in natural_results if "C2" in r]
    pooled_natural = {
        "delta_comet_mean": mean, "delta_comet_ci95": [lo, hi], "n_scored": len(all_deltas),
        "n_llm_calls_mean": round(sum(all_calls) / len(all_calls), 4) if all_calls else None,
        "n_passes_mean": round(sum(all_passes) / len(all_passes), 4) if all_passes else None,
    }

    # --- injected: fix_rate / true_regression_rate BY CATEGORY (not pooled), per language ---
    by_cat_lang = defaultdict(list)
    for r in injected_results:
        by_cat_lang[(r["category"], r["language_pair"])].append(r)

    category_table = []
    for cat in CATEGORY_MAP_INJECTED_TO_CHECKER:
        for pair in LANG_PAIRS:
            rows = by_cat_lang.get((cat, pair), [])
            if cat == "negation_polarity_flip":
                n_structural_nonattempt = sum(1 for r in rows if not r["C2"].get("c2_attempted", False))
                category_table.append({
                    "category": cat, "language_pair": pair, "n_rows": len(rows),
                    "fix_rate": NEGATION_LABEL, "true_regression_rate": NEGATION_LABEL,
                    "n_structural_nonattempt": n_structural_nonattempt,
                    "n_c2_attempted": len(rows) - n_structural_nonattempt,
                })
                continue
            attempted = [r for r in rows if r["C2"].get("c2_attempted", False) and r["C2"].get("fixed") is not None]
            fixed = [r["C2"]["fixed"] for r in attempted]
            regressed = [r["C2"]["true_regression"] for r in attempted]
            cert = [r["C2"]["certificate_achieved"] for r in attempted]
            passes = [r["C2"]["n_passes"] for r in attempted]
            calls = [r["C2"]["n_llm_calls"] for r in attempted]
            category_table.append({
                "category": cat, "language_pair": pair, "n_rows": len(rows), "n_c2_attempted": len(attempted),
                "n_structural_nonattempt": len(rows) - len(attempted),
                "fix_rate": round(sum(fixed) / len(fixed), 4) if fixed else None,
                "true_regression_rate": round(sum(regressed) / len(regressed), 4) if regressed else None,
                "certificate_rate": round(sum(cert) / len(cert), 4) if cert else None,
                "n_passes_mean": round(sum(passes) / len(passes), 4) if passes else None,
                "n_llm_calls_mean": round(sum(calls) / len(calls), 4) if calls else None,
            })

    # --- injected: pooled (all categories, all langs) EXCLUDING structural non-attempts from the rate denominators ---
    attempted_all = [r for r in injected_results if r["C2"].get("c2_attempted", False) and r["C2"].get("fixed") is not None]
    n_nonattempt_all = len(injected_results) - len(attempted_all)
    fixed_all = [r["C2"]["fixed"] for r in attempted_all]
    regressed_all = [r["C2"]["true_regression"] for r in attempted_all]
    cert_all = [r["C2"]["certificate_achieved"] for r in attempted_all]
    passes_all = [r["C2"]["n_passes"] for r in attempted_all]
    calls_all = [r["C2"]["n_llm_calls"] for r in attempted_all]
    pooled_injected = {
        "n_rows": len(injected_results), "n_c2_attempted": len(attempted_all),
        "n_structural_nonattempt_negation": n_nonattempt_all,
        "fix_rate": round(sum(fixed_all) / len(fixed_all), 4) if fixed_all else None,
        "true_regression_rate": round(sum(regressed_all) / len(regressed_all), 4) if regressed_all else None,
        "certificate_rate": round(sum(cert_all) / len(cert_all), 4) if cert_all else None,
        "n_passes_mean": round(sum(passes_all) / len(passes_all), 4) if passes_all else None,
        "n_llm_calls_mean": round(sum(calls_all) / len(calls_all), 4) if calls_all else None,
    }

    # --- CEGIS-narrowing signal: char_length_trend for multi-pass rows ---
    multi_pass_trends = []
    for r in injected_results:
        c2 = r.get("C2", {})
        if c2.get("n_passes", 0) >= 2:
            trend = [p["span_char_length_proxy"] for p in c2["pass_log"]]
            edit_trend = [p["edit_distance_chars"] for p in c2["pass_log"]]
            multi_pass_trends.append({
                "row_id": r["row_id"], "category": r["category"], "language_pair": r["language_pair"],
                "n_passes": c2["n_passes"], "char_length_trend": trend, "edit_distance_trend": edit_trend,
                "certificate_achieved": c2.get("certificate_achieved"),
            })
    for r in natural_results:
        c2 = r.get("C2", {})
        if c2.get("n_passes", 0) >= 2:
            trend = [p["span_char_length_proxy"] for p in c2["pass_log"]]
            edit_trend = [p["edit_distance_chars"] for p in c2["pass_log"]]
            multi_pass_trends.append({
                "row_id": r["row_id"], "category": "natural", "language_pair": r["language_pair"],
                "n_passes": c2["n_passes"], "char_length_trend": trend, "edit_distance_trend": edit_trend,
                "certificate_achieved": c2.get("certificate_achieved"),
            })
    n_narrowing = sum(
        1 for t in multi_pass_trends
        if len(t["edit_distance_trend"]) >= 2 and t["edit_distance_trend"][-1] <= t["edit_distance_trend"][0]
    )
    n_diverging = len(multi_pass_trends) - n_narrowing
    cegis_signal = {
        "n_multi_pass_rows": len(multi_pass_trends),
        "n_edit_distance_narrowing_or_flat": n_narrowing,
        "n_edit_distance_diverging": n_diverging,
        "interpretation": (
            "A row 'narrows' if its LAST pass's edit distance from the prior-pass sentence is <= its FIRST "
            "pass's edit distance (a weak proxy for convergence toward a stable repair, since exact span "
            "boundaries drift as the whole sentence is rewritten each pass). This is descriptive, not a claim "
            "of formal CEGIS convergence -- see multi_pass_trends for the raw per-row sequences."
        ),
    }

    # --- gain-to-edit ratio (natural): DeltaCOMET gain per unit of edit volume ---
    gain_to_edit = []
    for pair in LANG_PAIRS:
        m = per_pair[pair]
        if m["edit_volume_mean"] and m["edit_volume_mean"] > 0:
            gain_to_edit.append({"language_pair": pair, "delta_comet_per_edit_volume": round(m["delta_comet_mean"] / m["edit_volume_mean"], 6)})

    return {
        "natural_per_language_pair": dict(per_pair),
        "natural_pooled": pooled_natural,
        "injected_category_table": category_table,
        "injected_pooled": pooled_injected,
        "cegis_narrowing_signal": cegis_signal,
        "multi_pass_trends": multi_pass_trends,
        "gain_to_edit_ratio": gain_to_edit,
    }


def build_comparison_table(c2_metrics: dict) -> dict:
    """Step 6: side-by-side B, D, C1 (loaded from PRIOR_RUN_PATH/method_out.json)
    vs C2 (this run), same metrics, same row population, same language pairs."""
    prior_path = PRIOR_RUN_PATH / "method_out.json"
    if not prior_path.exists():
        logger.warning(f"Prior method_out.json not found at {prior_path}; comparison table will only contain C2.")
        return {"prior_run_found": False, "conditions": {"C2": c2_metrics["natural_pooled"] | c2_metrics["injected_pooled"]}}
    prior = json.loads(prior_path.read_text())
    prior_pooled = prior["metadata"]["metrics"]["pooled"]
    table = {}
    for cond in ["B", "D", "C1"]:
        p = prior_pooled.get(cond, {})
        table[cond] = {
            "delta_comet_mean": p.get("delta_comet_mean"), "delta_comet_ci95": p.get("delta_comet_ci95"),
            "n_natural_scored": p.get("n_scored"), "fix_rate_pooled": p.get("fix_rate"),
            "true_regression_rate_pooled": p.get("true_regression_rate"),
            "n_injected_scored": p.get("n_injected_scored"),
        }
    table["C2"] = {
        "delta_comet_mean": c2_metrics["natural_pooled"]["delta_comet_mean"],
        "delta_comet_ci95": c2_metrics["natural_pooled"]["delta_comet_ci95"],
        "n_natural_scored": c2_metrics["natural_pooled"]["n_scored"],
        "fix_rate_pooled": c2_metrics["injected_pooled"]["fix_rate"],
        "true_regression_rate_pooled": c2_metrics["injected_pooled"]["true_regression_rate"],
        "n_injected_scored": c2_metrics["injected_pooled"]["n_c2_attempted"],
        "certificate_rate_pooled": c2_metrics["injected_pooled"]["certificate_rate"],
        "note": (
            "C2's n_injected_scored excludes structural non-attempts (negation_polarity_flip deletions), "
            f"n={c2_metrics['injected_pooled']['n_structural_nonattempt_negation']} -- B/D/C1's n_injected_scored "
            "(240) includes those rows with fixed=False by construction (see B/D/C1's own "
            "injected_pool_localization_protocol_caveat), so a raw fix_rate comparison against B/D is slightly "
            "conservative for B/D (their denominator is larger) -- reported as-is, not adjusted, and this note "
            "documents the asymmetry rather than silently normalizing it away."
        ),
    }
    return {"prior_run_found": True, "conditions": table, "success_criterion_b_note": (
        "Success criterion (b) asks whether C2 vs C1 improves regression rate and whether C2 improves "
        "DeltaCOMET over D. Read true_regression_rate_pooled and delta_comet_mean directly off this table "
        "for C1/D/C2."
    )}


## Main — Phase 0/1 setup

The first half of `method.py`'s `main()`: resource limits, row selection, the Stanza NER resource
download (needed because this notebook, like the original workspace, has no pre-cached
`~/stanza_resources`), and the Phase-1 checker validation against the `checker_validation_heldout`
rows.


In [ ]:
t0 = time.time()
setup_resource_limits()
rng = random.Random(SEED)

natural_all = by_dataset["wmt25_task3_natural"]
injected_all = by_dataset["injected_error_augmentation"]

natural_rows = stratified_sample_natural(natural_all, rng)
injected_pool_rows = stratified_sample_injected(injected_all, rng, "experimental_pool", N_INJECTED_PER_CELL)
injected_heldout_rows = stratified_sample_injected(injected_all, rng, "checker_validation_heldout", None)

row_id_check = check_row_id_match(natural_rows, injected_pool_rows, injected_heldout_rows)

(WORKDIR / "selected_rows.json").write_text(json.dumps({
    "seed": SEED,
    "natural_row_ids": [r["_row_id"] for r in natural_rows],
    "injected_pool_row_ids": [r["_row_id"] for r in injected_pool_rows],
    "injected_heldout_row_ids": [r["_row_id"] for r in injected_heldout_rows],
}, indent=2))
logger.info(
    f"Selected {len(natural_rows)} natural, {len(injected_pool_rows)} injected-pool, "
    f"{len(injected_heldout_rows)} injected-heldout rows"
)

del natural_all, injected_all
gc.collect()

# --- ensure Stanza NER resources are present (this workspace has no
# pre-cached ~/stanza_resources) -- download_method=None inside checker.py's
# NamedEntityDetector requires local resources to already exist, so this
# downloads them once, up front, without modifying checker.py itself. ---
try:
    import stanza

    for stanza_lang in ("ru", "uk"):
        stanza.download(stanza_lang, processors="tokenize,ner", verbose=False)
    logger.info("Stanza ru/uk tokenize,ner resources downloaded.")
except Exception as e:
    logger.error(f"Stanza resource download failed: {e}. named_entity detection will error and that (lang, "
                  "named_entity) cell will be excluded from validation (recall/precision default to 0).")

# --- Phase 1: build + validate checker ---
checker_unrestricted = Checker(excluded_cells=set())
logger.info("Running full checker validation on injected_heldout fold...")
validation = validate_checker(checker_unrestricted, injected_heldout_rows)
excluded = excluded_cells_from_validation(validation)
logger.info(f"Checker validation: {json.dumps(validation, indent=2)}")
logger.info(f"Excluded cells (recall<0.5 or precision<0.5): {sorted(excluded)}")
checker = Checker(excluded_cells=excluded)

n_negation_injected = sum(1 for r in injected_pool_rows if is_negation_deletion_row(r))
logger.info(f"negation_polarity_flip deletion rows (structural non-attempt): {n_negation_injected}")


## Main — cost estimate + the C2 sweep

This is where the real OpenRouter calls happen: a small cost-estimate probe, then the full C2
iterative-repair sweep over every selected row, both hard-stopped against `SUB_BUDGET_USD`.


In [ ]:
# --- Step 4: pre-sweep cost estimate ---
# (notebook-context fix: `await` directly instead of asyncio.run() -- Jupyter's
# kernel already runs its own event loop, which asyncio.run() rejects)
cost_estimate = await estimate_cost(natural_rows, injected_pool_rows, checker, excluded)
logger.info(f"Cost estimate decision: {cost_estimate['decision']}")

# --- Steps 3-4: run C2 with cumulative-spend hard stop at SUB_BUDGET_USD ---
ledger = CostLedger(cap_usd=MAX_USD_BUDGET)
natural_results, injected_results = await run_pipeline(natural_rows, injected_pool_rows, checker, excluded, ledger)
logger.info(f"Pipeline complete. Cost ledger: {ledger.summary()}")
if ledger.spent_usd > SUB_BUDGET_USD:
    logger.warning(f"Final spend ${ledger.spent_usd:.4f} exceeded the pre-declared sub-budget ${SUB_BUDGET_USD:.2f}.")


## Main — scoring + summary output

COMET scoring (proxy fallback, as noted above), then `summarize_c2` / `build_comparison_table` to
assemble the same `method_out.json`-shaped output the original script writes.


In [ ]:
# --- Phase 5: COMET scoring on natural rows (C2 vs original only) ---
comet_model = try_load_comet()
comet_available = comet_model is not None
natural_comet = {p: [] for p in LANG_PAIRS}
row_source_cache = {r["_row_id"]: r["input"] for r in natural_rows}

if comet_available:
    for pair in LANG_PAIRS:
        pair_rows = [r for r in natural_results if r["language_pair"] == pair]
        sources = [row_source_cache[r["row_id"]] for r in pair_rows]
        originals = [r["original"] for r in pair_rows]
        edited = [r["C2"]["output"] if "C2" in r and "output" in r["C2"] else r["original"] for r in pair_rows]
        orig_scores = comet_score_batch(comet_model, sources, originals)
        edited_scores = comet_score_batch(comet_model, sources, edited)
        deltas = [e - o for e, o in zip(edited_scores, orig_scores)]
        natural_comet[pair] = deltas
        del sources, originals, edited, orig_scores, edited_scores
        gc.collect()
    del comet_model
    gc.collect()
    import torch

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    for pair in LANG_PAIRS:
        pair_rows = [r for r in natural_results if r["language_pair"] == pair]
        sources = [row_source_cache[r["row_id"]] for r in pair_rows]
        originals = [r["original"] for r in pair_rows]
        edited = [r["C2"]["output"] if "C2" in r and "output" in r["C2"] else r["original"] for r in pair_rows]
        orig_scores = [proxy_qe_score(s, o) for s, o in zip(sources, originals)]
        edited_scores = [proxy_qe_score(s, e) for s, e in zip(sources, edited)]
        natural_comet[pair] = [e - o for e, o in zip(edited_scores, orig_scores)]

c2_metrics = summarize_c2(natural_results, injected_results, natural_comet, rng)
comparison_table = build_comparison_table(c2_metrics)

elapsed = time.time() - t0
logger.info(f"Total runtime: {elapsed:.1f}s.")


## Results

The headline numbers from the full run were: fix_rate (injected-pool, among attempted rows) jumped
from B/D's 0.7375 and C1's 0.4542 to C2's **0.9889**, and true_regression_rate dropped from C1's 0.125
to C2's **0.0722** — but ΔCOMET on natural rows did *not* improve (C2 -0.0166 vs D's -0.0157), i.e.
iteration alone drives fix rate up and regression down without helping (and if anything, at this tiny
scale, without changing) translation quality. This cell prints and plots the same numbers computed
above from the mini demo subset — with only a handful of rows, individual outcomes dominate and won't
match the full-scale statistics, but the same tables and figures the full run produces are shown here.


In [ ]:
print("=" * 70)
print("COST LEDGER")
print("=" * 70)
for k, v in ledger.summary().items():
    print(f"  {k:>16}: {v}")

print()
print("=" * 70)
print("NATURAL ROWS -- pooled + per-language-pair (delta-COMET / proxy)")
print("=" * 70)
print(f"{'scope':<12}{'delta_mean':>12}{'ci_lo':>10}{'ci_hi':>10}{'n':>5}{'n_calls':>9}{'n_passes':>10}")
pn = c2_metrics["natural_pooled"]
print(f"{'pooled':<12}{pn['delta_comet_mean']:>12.4f}{pn['delta_comet_ci95'][0]:>10.4f}{pn['delta_comet_ci95'][1]:>10.4f}{pn['n_scored']:>5}{(pn['n_llm_calls_mean'] or 0):>9.2f}{(pn['n_passes_mean'] or 0):>10.2f}")
for pair, m in c2_metrics["natural_per_language_pair"].items():
    print(f"{pair:<12}{m['delta_comet_mean']:>12.4f}{m['delta_comet_ci95'][0]:>10.4f}{m['delta_comet_ci95'][1]:>10.4f}{m['n_scored']:>5}{(m['n_llm_calls_mean'] or 0):>9.2f}{(m['n_passes_mean'] or 0):>10.2f}")

print()
print("=" * 70)
print("INJECTED ROWS -- by category x language pair (fix_rate / true_regression_rate / certificate_rate)")
print("=" * 70)
print(f"{'category':<28}{'pair':<10}{'n':>4}{'fix_rate':>12}{'regress':>10}{'cert':>8}")
for row in c2_metrics["injected_category_table"]:
    fr = row["fix_rate"] if isinstance(row["fix_rate"], str) else f"{row['fix_rate']:.3f}" if row["fix_rate"] is not None else "n/a"
    tr = row["true_regression_rate"] if isinstance(row["true_regression_rate"], str) else f"{row['true_regression_rate']:.3f}" if row["true_regression_rate"] is not None else "n/a"
    ce = f"{row['certificate_rate']:.3f}" if row.get("certificate_rate") is not None else "n/a"
    print(f"{row['category']:<28}{row['language_pair']:<10}{row['n_rows']:>4}{fr:>12}{tr:>10}{ce:>8}")

print()
print("=" * 70)
print("INJECTED ROWS -- pooled")
print("=" * 70)
for k, v in c2_metrics["injected_pooled"].items():
    print(f"  {k:>32}: {v}")

print()
print("=" * 70)
print("B / D / C1 (prior run, if shipped) vs C2 (this notebook)")
print("=" * 70)
print(json.dumps(comparison_table, indent=2, default=str)[:3000])

print()
print("=" * 70)
print("CEGIS-narrowing signal (descriptive, multi-pass rows only)")
print("=" * 70)
print(json.dumps(c2_metrics["cegis_narrowing_signal"], indent=2))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Panel 1: fix_rate / true_regression_rate per attempted injected category (this demo) ---
cats_plot = [r for r in c2_metrics["injected_category_table"] if isinstance(r["fix_rate"], float)]
labels = [f"{r['category'].replace('_', ' ')}\n[{r['language_pair']}]" for r in cats_plot]
fix_rates = [r["fix_rate"] for r in cats_plot]
regress_rates = [r["true_regression_rate"] for r in cats_plot]
x = range(len(cats_plot))
w = 0.35
axes[0].bar([i - w / 2 for i in x], fix_rates, width=w, label="fix_rate", color="#2a9d8f")
axes[0].bar([i + w / 2 for i in x], regress_rates, width=w, label="true_regression_rate", color="#e76f51")
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(labels, fontsize=7, rotation=45, ha="right")
axes[0].set_ylim(0, 1.05)
axes[0].set_title("C2 outcomes by category (this demo)")
axes[0].legend(fontsize=8)

# --- Panel 2: this demo's C2 vs the full run's B/D/C1/C2 pooled numbers ---
if comparison_table.get("prior_run_found"):
    conds = list(comparison_table["conditions"].keys())
    fr_vals = [comparison_table["conditions"][c]["fix_rate_pooled"] for c in conds]
    axes[1].bar(conds, fr_vals, color=["#264653", "#2a9d8f", "#e9c46a", "#e76f51"][: len(conds)])
    axes[1].set_ylim(0, 1.05)
    axes[1].set_title("Pooled fix_rate: B/D/C1 (full run) vs C2")
else:
    pooled = c2_metrics["injected_pooled"]
    axes[1].bar(["C2 (this demo)"], [pooled["fix_rate"] or 0], color="#2a9d8f", width=0.4)
    axes[1].set_xlim(-1, 1)
    axes[1].set_ylim(0, 1.05)
    axes[1].set_title("Pooled fix_rate (this demo)\n(prior B/D/C1 run not shipped with this notebook)")

# --- Panel 3: delta-COMET (or proxy) per language pair, this demo ---
pairs_plot = list(c2_metrics["natural_per_language_pair"].keys())
means = [c2_metrics["natural_per_language_pair"][p]["delta_comet_mean"] for p in pairs_plot]
los = [c2_metrics["natural_per_language_pair"][p]["delta_comet_ci95"][0] for p in pairs_plot]
his = [c2_metrics["natural_per_language_pair"][p]["delta_comet_ci95"][1] for p in pairs_plot]
errs = [[m - lo for m, lo in zip(means, los)], [hi - m for m, hi in zip(means, his)]]
axes[2].bar(pairs_plot, means, yerr=errs, capsize=5, color="#457b9d")
axes[2].axhline(0, color="black", linewidth=0.8)
metric_name = "delta-COMET" if comet_available else "delta-COMET-proxy"
axes[2].set_title(f"Natural rows: {metric_name} (this demo, 95% CI)")

plt.tight_layout()
plt.savefig("results_summary.png", dpi=130)
plt.show()
